In [6]:
from manim import *
config.media_width = "75%"  # Größe der Anzeige im Notebook

class KLFeasibleSet(Scene):
    def construct(self):
        # Achsen
        axes = Axes(
            x_range=[0, 1, 0.2],
            y_range=[0, 1, 0.2],
            x_length=6,
            y_length=6,
            axis_config={"color": WHITE}
        ).add_coordinates()

        x_label = axes.get_x_axis_label(Tex("$q_4 + q_5$"), edge=RIGHT, direction=DOWN)
        y_label = axes.get_y_axis_label(Tex("Other Constraint"), edge=UP, direction=LEFT)

        # Ursprünglicher zulässiger Bereich (großes Quadrat)
        original_region = Polygon(
            axes.c2p(0.2, 0.2),
            axes.c2p(0.8, 0.2),
            axes.c2p(0.8, 0.8),
            axes.c2p(0.2, 0.8),
            color=BLUE,
            fill_opacity=0.4
        )

        # Eingeschränkter Bereich durch KL-Bedingung (kleineres Quadrat)
        kl_region = Polygon(
            axes.c2p(0.35, 0.35),
            axes.c2p(0.65, 0.35),
            axes.c2p(0.65, 0.65),
            axes.c2p(0.35, 0.65),
            color=YELLOW,
            fill_opacity=0.6
        )

        # Labels
        original_label = Tex("Feasible Set (No KL)").next_to(original_region, UP).set_color(BLUE)
        kl_label = Tex(r"Feasible Set (KL $\leq$ $\theta$)").next_to(kl_region, DOWN).set_color(YELLOW)

        # Aufbau
        self.play(Create(axes), Write(x_label), Write(y_label))
        self.play(FadeIn(original_region), FadeIn(original_label))
        self.wait(1)
        self.play(FadeIn(kl_region), FadeIn(kl_label))
        self.wait(2)

        # Pfeil zur Schrumpfung
        arrow = Arrow(start=axes.c2p(0.8, 0.8), end=axes.c2p(0.65, 0.65), buff=0.1, color=WHITE)
        self.play(GrowArrow(arrow))
        self.wait(2)
        self.play(FadeOut(arrow), FadeOut(original_label), FadeOut(kl_label))
        self.wait(1)

        # KL Erklärung
        kl_text = Tex(r"KL Constraint: $\frac{1}{\log 2} \sum_i q_i \log \frac{q_i}{r_i} \leq \theta$")
        kl_text.to_edge(DOWN)
        self.play(Write(kl_text))
        self.wait(3)

        self.play(*[FadeOut(mob) for mob in self.mobjects])


In [15]:
%%manim -v WARNING -qm DAGWithConfounder

from manim import *

class DAGWithConfounder(Scene):
    def construct(self):
        # Create nodes (closer together, absolute positions)
        X_circle = Circle(radius=0.6, color=WHITE)
        X = VGroup(X_circle, MathTex("X").move_to(X_circle.get_center())).move_to([-2.5, 0, 0])

        Y_circle = Circle(radius=0.6, color=WHITE)
        Y = VGroup(Y_circle, MathTex("Y").move_to(Y_circle.get_center())).move_to([2.5, 0, 0])

        U_raw = Circle(radius=0.6, color=WHITE)
        U = DashedVMobject(U_raw)
        U_group = VGroup(U, MathTex("U").move_to(U.get_center())).move_to([0, 2.5, 0])

        # Arrows
        arrow_XY = Arrow(X.get_right(), Y.get_left(), buff=0)
        arrow_UX = Arrow(U_group.get_bottom(), X.get_top(), buff=0)
        arrow_UY = Arrow(U_group.get_bottom(), Y.get_top(), buff=0)

        # Group all objects and set full scale and position from the beginning
        all_objects = VGroup(X, Y, U_group, arrow_XY, arrow_UX, arrow_UY)
        all_objects.scale(1.6).move_to(ORIGIN)

        # Step 1: Show initial graph
        self.play(FadeIn(X), FadeIn(Y), FadeIn(U_group))
        self.play(GrowArrow(arrow_XY), GrowArrow(arrow_UX), GrowArrow(arrow_UY))
        self.wait(1)

Manim Community v0.19.0

In [10]:

%%manim -v WARNING -qm ConfoundingBreaksIdentifiability
from manim import *

class ConfoundingBreaksIdentifiability(Scene):
    def construct(self):
        # DAG: X -> Y
        x = Circle(radius=0.4, color=BLUE).shift(LEFT * 2)
        x_label = Tex("X").move_to(x.get_center())
        y = Circle(radius=0.4, color=BLUE).shift(RIGHT * 2)
        y_label = Tex("Y").move_to(y.get_center())
        arrow_xy = Arrow(start=x.get_right(), end=y.get_left(), buff=0.05)

        dag_group = VGroup(x, y, x_label, y_label, arrow_xy)

        # Equation: observed difference = ATE
        eq = MathTex(r"\mathbb{E}[Y \mid X=1] - \mathbb{E}[Y \mid X=0] = \mathrm{ATE}")
        eq.scale(0.9).shift(DOWN * 2)

        # Assumption text
        assumption = Tex(r"Assume: $X \perp (Y_0, Y_1)$")
        assumption.scale(0.8).next_to(eq, UP)

        # Animate setup
        self.play(FadeIn(dag_group))
        self.wait(0.5)
        self.play(Write(assumption))
        self.play(Write(eq))
        self.wait(1)

        # Add confounder U
        u = Circle(radius=0.4, color=RED).shift(UP * 2)
        u_label = Tex("U").move_to(u.get_center())
        arrow_ux = Arrow(start=u.get_bottom(), end=x.get_top(), buff=0.05)
        arrow_uy = Arrow(start=u.get_bottom(), end=y.get_top(), buff=0.05)

        self.play(FadeIn(u), FadeIn(u_label))
        self.play(GrowArrow(arrow_ux), GrowArrow(arrow_uy))
        self.wait(1)

        # Cross out the independence assumption
        cross_assump = Cross(assumption, stroke_color=RED, stroke_width=4)
        self.play(Create(cross_assump))

        # Cross out the equal sign in the equation
        eq_equals_idx = 14  # Adjust if you change the equation
        eq_equals_char = eq[0][eq_equals_idx]
        cross_eq = Line(
            eq_equals_char.get_top() + UP * 0.05,
            eq_equals_char.get_bottom() + DOWN * 0.05,
            color=RED,
            stroke_width=4
        )
        self.play(Create(cross_eq))
        self.wait(2)

        self.play(*[FadeOut(mob) for mob in self.mobjects])


Manim Community v0.19.0

In [7]:
%load_ext manim


The manim module is not an IPython extension.


In [10]:
%%manim -v WARNING -qm ATEBackdoorImproved

from manim import *


class ATEBackdoorImproved(Scene):
    def construct(self):
        # --- ATE (potential outcomes) formula ---
        ate_po = MathTex(r"\mathrm{ATE} = \mathbb{E}[Y_1] - \mathbb{E}[Y_0]").scale(0.9).to_edge(UP)

        # --- Backdoor adjustment formula ---
        ate_backdoor = MathTex(
            r"\mathrm{ATE} = \sum_u \left( \mathbb{E}[Y \mid X=1, U=u] - \mathbb{E}[Y \mid X=0, U=u] \right) P(U=u)"
        ).scale(0.75).next_to(ate_po, DOWN, buff=1)

        # --- DAG node positions ---
        x_pos = LEFT * 2
        y_pos = RIGHT * 2
        u_pos = UP * 1.8  # slightly higher
        dag_y_shift = DOWN * 0.9  # move everything upward

        # --- DAG nodes ---
        u_raw = Circle(radius=0.5, color=WHITE)
        u_dashed = DashedVMobject(u_raw).move_to(u_pos + dag_y_shift)
        u_label = Tex("U").move_to(u_dashed.get_center())

        x = Circle(radius=0.5, color=WHITE).move_to(x_pos + dag_y_shift)
        x_label = Tex("X").move_to(x.get_center())

        y = Circle(radius=0.5, color=WHITE).move_to(y_pos + dag_y_shift)
        y_label = Tex("Y").move_to(y.get_center())

        # --- DAG edges ---
        arrow_ux = Arrow(start=u_dashed.get_bottom(), end=x.get_top(), buff=0.1)
        arrow_uy = Arrow(start=u_dashed.get_bottom(), end=y.get_top(), buff=0.1)
        arrow_xy = Arrow(start=x.get_right(), end=y.get_left(), buff=0.1)

        # --- Group DAG elements ---
        dag_group = VGroup(u_dashed, u_label, x, x_label, y, y_label, arrow_ux, arrow_uy, arrow_xy)

        # --- Start animation ---
        self.play(Write(ate_po))
        self.wait(1)

        self.play(FadeIn(u_dashed), FadeIn(u_label),
                  FadeIn(x), FadeIn(x_label),
                  FadeIn(y), FadeIn(y_label),
                  GrowArrow(arrow_ux), GrowArrow(arrow_uy), GrowArrow(arrow_xy))
        self.wait(1)

        # Transform dashed U into solid U
        u_solid = u_raw.copy().move_to(u_dashed.get_center())
        self.play(ReplacementTransform(u_dashed, u_solid))

        self.play(TransformMatchingTex(ate_po, ate_backdoor))
        self.wait(1)

        
        self.wait(2)



Manim Community v0.19.0